# BERTopic Topic Model Training
## Mamina Baby Spa

Notebook ini melatih model topic clustering dari pesan WhatsApp customer. Alur utamanya mengikuti `backend/scripts/train_topic_model.py` agar hasil notebook konsisten dengan pipeline backend.

Default output disimpan ke `backend/models/topic_model_candidate` supaya tidak langsung menimpa model topic produksi.

## Cell 1: Install Dependencies

Jalankan cell ini hanya jika environment notebook belum punya dependency backend.

In [ ]:
# Optional. Uncomment jika package belum tersedia di environment notebook.
# !pip install -r ../backend/requirements.txt

## Cell 2: Setup Path, Environment, and Imports

In [ ]:
import os
import sys
import json
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists() and (PROJECT_ROOT.parent / "backend").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# Default mengikuti backend/docker/docker-compose.yml saat notebook dijalankan dari host/Windows.
os.environ.setdefault("DB_HOST", "127.0.0.1")
os.environ.setdefault("DB_PORT", "5432")
os.environ.setdefault("DB_NAME", "churn_db")
os.environ.setdefault("DB_USER", "mamina")
os.environ.setdefault("DB_PASSWORD", "mamina_password")
os.environ.setdefault(
    "DATABASE_URL",
    f"postgresql://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}",
)
os.environ["FLASK_ENV"] = "development"
os.environ["LOG_FILE"] = str(BACKEND_DIR / "logs" / "notebook_topic_training.log")
os.environ["MODEL_PATH"] = str(BACKEND_DIR / "models" / "multimodal_model.pkl")
os.environ["SCALER_PATH"] = str(BACKEND_DIR / "models" / "scaler.pkl")
os.environ["FEATURE_META_PATH"] = str(BACKEND_DIR / "models" / "features.json")
os.environ["SHAP_EXPLAINER_PATH"] = str(BACKEND_DIR / "models" / "shap_explainer.pkl")

from app import create_app, db
from app.models.topic import Topic, ModelVersion
from app.services.topic_service import TopicService
from scripts.train_topic_model import (
    DEFAULT_TARGET_TOPICS,
    deduplicate_texts,
    human_topic_name,
    is_trainable_text,
    load_texts_from_csv,
    load_texts_from_db,
    normalize_text,
    register_topic_model_version,
    reset_output_path,
    upsert_topics,
    validate_topic_runtime,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Backend dir : {BACKEND_DIR}")
print(f"Database    : {os.environ['DB_USER']}@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}")

## Cell 3: Training Configuration

In [ ]:
SOURCE = "db"  # "db" atau "csv"
CSV_PATH = PROJECT_ROOT / "whatsapp_messages.csv"
DIRECTION = "inbound"
MIN_CHARS = 15
MIN_DOCS = 50
LIMIT = None  # contoh: 1000 untuk eksperimen cepat
TARGET_TOPICS = DEFAULT_TARGET_TOPICS
AUTO_TUNE = True
INCLUDE_PROVISIONAL = False
REPLACE_TOPICS = True

# Default candidate agar tidak langsung menimpa backend/models/topic_model produksi.
OUTPUT_DIR_NAME = "topic_model_candidate"
OUTPUT_PATH = BACKEND_DIR / "models" / OUTPUT_DIR_NAME
REGISTERED_MODEL_PATH = f"models/{OUTPUT_DIR_NAME}"
OVERWRITE_OUTPUT = True

MODEL_VERSION = f"bertopic_notebook_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

config = {
    "source": SOURCE,
    "csv_path": str(CSV_PATH),
    "direction": DIRECTION,
    "min_chars": MIN_CHARS,
    "min_docs": MIN_DOCS,
    "limit": LIMIT,
    "target_topics": TARGET_TOPICS,
    "auto_tune": AUTO_TUNE,
    "include_provisional": INCLUDE_PROVISIONAL,
    "replace_topics": REPLACE_TOPICS,
    "output_path": str(OUTPUT_PATH),
    "registered_model_path": REGISTERED_MODEL_PATH,
    "model_version": MODEL_VERSION,
}
print(json.dumps(config, indent=2))

## Cell 4: Connect Flask App and Database

In [ ]:
app = create_app(os.environ.get("FLASK_ENV", "development"))

with app.app_context():
    row = db.session.execute(db.text("SELECT current_database(), current_user")).one()
    topic_count = Topic.query.count()
    version_count = ModelVersion.query.count()

print(f"Database connected successfully: {row[1]}@{row[0]}")
print(f"Existing topics       : {topic_count}")
print(f"Existing model version: {version_count}")

## Cell 5: Load and Preview Training Corpus

In [ ]:
with app.app_context():
    if SOURCE == "db":
        texts = load_texts_from_db(
            direction=DIRECTION,
            min_chars=MIN_CHARS,
            limit=LIMIT,
            trusted_only=not INCLUDE_PROVISIONAL,
        )
    elif SOURCE == "csv":
        texts = load_texts_from_csv(
            csv_path=str(CSV_PATH),
            direction=DIRECTION,
            min_chars=MIN_CHARS,
            limit=LIMIT,
        )
    else:
        raise ValueError(f"Unknown SOURCE: {SOURCE}")

print(f"Loaded trainable texts: {len(texts)}")
if len(texts) < MIN_DOCS:
    raise RuntimeError(
        f"Need at least {MIN_DOCS} trainable texts, got {len(texts)}. "
        "Import more WhatsApp data or lower MIN_DOCS for experimentation."
    )

preview_df = pd.DataFrame({"text": texts[:20]})
preview_df

## Cell 6: Corpus Diagnostics

In [ ]:
lengths = pd.Series([len(text) for text in texts], name="chars")
diagnostics = lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame().T
print(f"Unique normalized texts: {len(deduplicate_texts(texts))}")
diagnostics

## Cell 6.5: Corpus Visualization

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(lengths, bins=30, kde=True, color="#2563eb")
plt.axvline(MIN_CHARS, color="red", linestyle="--", label=f"MIN_CHARS = {MIN_CHARS}")
plt.title("Distribusi Panjang Pesan Training")
plt.xlabel("Jumlah Karakter")
plt.ylabel("Jumlah Pesan")
plt.legend()
plt.tight_layout()
plt.show()


## Cell 7: Initialize Topic Service and Encode Corpus

In [ ]:
validate_topic_runtime()

service = TopicService()
service.load_model(version=MODEL_VERSION)

print("Encoding corpus...")
embeddings = service.embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
)
print(f"Embeddings shape: {embeddings.shape}")

## Cell 8: Optional Clustering Auto-Tune

In [ ]:
tuning_result = None

if AUTO_TUNE:
    tuning_result = service.tune_clustering(
        embeddings,
        max_topics=TARGET_TOPICS,
    )
    tuning_trials_df = pd.DataFrame(tuning_result["trials"])
    best_params = tuning_result["best_params"]
    print(f"Selected clustering parameters: {best_params}")
    service.load_model(version=MODEL_VERSION, clustering_params=best_params)
else:
    tuning_trials_df = pd.DataFrame()
    print("AUTO_TUNE disabled; using TopicService default clustering parameters.")

tuning_trials_df

## Cell 9: Train BERTopic Model

In [ ]:
train_result = service.train(
    texts,
    embeddings=embeddings,
    target_topics=TARGET_TOPICS,
)

topics_list = list(train_result.get("topics") or [])
topic_info_df = pd.DataFrame(train_result.get("topic_info") or [])
print(f"Topic assignments: {len(topics_list)}")
print(f"Topic rows       : {len(topic_info_df)}")
topic_info_df.head(20)

## Cell 9.5: Topic Distribution Visualization

In [ ]:
plot_topic_df = topic_info_df.copy()
if not plot_topic_df.empty and "Topic" in plot_topic_df.columns:
    plot_topic_df["topic_label"] = plot_topic_df["Topic"].apply(
        lambda topic: "Outlier (-1)" if int(topic) == -1 else f"Topic {int(topic)}"
    )
    plot_topic_df = plot_topic_df.sort_values("Count", ascending=False).head(20)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=plot_topic_df, x="Count", y="topic_label", palette="viridis")
    plt.title("Distribusi Dokumen per Topic")
    plt.xlabel("Jumlah Dokumen")
    plt.ylabel("Topic")
    plt.tight_layout()
    plt.show()
else:
    print("topic_info_df kosong atau tidak memiliki kolom Topic.")


## Cell 10: Evaluate Clustering Quality

In [ ]:
if topics_list:
    eval_metrics = service.evaluate(
        texts=texts,
        topics=topics_list,
        embeddings=embeddings,
    )
    if tuning_result:
        eval_metrics["tuning"] = tuning_result
else:
    eval_metrics = {"evaluation_error": "No topic assignments returned from training."}

print(json.dumps(eval_metrics, indent=2, default=str))
pd.DataFrame([{
    "n_docs": eval_metrics.get("n_docs"),
    "n_topics_found": eval_metrics.get("n_topics_found"),
    "n_outliers": eval_metrics.get("n_outliers"),
    "outlier_rate": eval_metrics.get("outlier_rate"),
    "topic_diversity": eval_metrics.get("topic_diversity"),
    "silhouette_score": eval_metrics.get("silhouette_score"),
}])

## Cell 10.5: Evaluation Metrics Visualization

In [ ]:
metric_plot_df = pd.DataFrame([
    {"metric": "Outlier Rate", "value": eval_metrics.get("outlier_rate")},
    {"metric": "Topic Diversity", "value": eval_metrics.get("topic_diversity")},
    {"metric": "Silhouette Score", "value": eval_metrics.get("silhouette_score")},
]).dropna()

if not metric_plot_df.empty:
    plt.figure(figsize=(9, 5))
    ax = sns.barplot(data=metric_plot_df, x="metric", y="value", palette=["#ef4444", "#2563eb", "#16a34a"][:len(metric_plot_df)])
    ax.set_title("Ringkasan Evaluasi BERTopic")
    ax.set_xlabel("")
    ax.set_ylabel("Nilai")
    ax.set_ylim(min(-0.1, metric_plot_df["value"].min() - 0.05), 1.05)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=3)
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada metrik numerik yang bisa divisualisasikan.")

warnings = eval_metrics.get("evaluation_warnings") or []
if warnings:
    print("Evaluation warnings:")
    for warning in warnings:
        print(f"- {warning}")


## Cell 11: Save Model and Persist Topic Metadata

In [ ]:
reset_output_path(str(OUTPUT_PATH), overwrite=OVERWRITE_OUTPUT)

if not service.save_model(str(OUTPUT_PATH)):
    raise RuntimeError(f"Failed to save topic model to {OUTPUT_PATH}")

service.model_version = MODEL_VERSION

with app.app_context():
    saved_topics = upsert_topics(
        topic_service=service,
        model_version=MODEL_VERSION,
        replace_topics=REPLACE_TOPICS,
    )
    register_topic_model_version(
        model_version=MODEL_VERSION,
        output_path=REGISTERED_MODEL_PATH,
        eval_metrics=eval_metrics,
    )

print(f"Saved model directory : {OUTPUT_PATH}")
print(f"Registered model path : {REGISTERED_MODEL_PATH}")
print(f"Model version         : {MODEL_VERSION}")
print(f"Saved topic rows      : {saved_topics}")

## Cell 12: Inspect Saved Topics

In [ ]:
with app.app_context():
    rows = (
        Topic.query
        .filter(Topic.model_version == MODEL_VERSION)
        .order_by(Topic.topic_idx.asc())
        .all()
    )

saved_topics_df = pd.DataFrame([
    {
        "topic_idx": row.topic_idx,
        "name": row.name,
        "top_keywords": ", ".join(row.top_keywords or []),
        "model_version": row.model_version,
    }
    for row in rows
])
saved_topics_df

## Notes

- Untuk menjadikan hasil notebook sebagai model produksi, ubah `OUTPUT_DIR_NAME` menjadi `topic_model`, atau salin hasil candidate ke `backend/models/topic_model` setelah evaluasi diterima.
- Backend Docker membaca model dari `/app/models/topic_model`, yang terhubung ke folder host `backend/models/topic_model`.
- Metrik evaluasi tersimpan di tabel `model_versions` dengan `notes = 'BERTopic clustering model'`.